In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"

sales = pd.read_parquet(DATA / "sales.parquet")
sales["series"] = sales["store_id"] + "/" + sales["item_id"]
sales["is_zero"] = sales["units"] == 0
print(sales.shape)
sales.dtypes

In [ ]:
profile = sales.groupby("series").agg(
    start=("date", "min"),
    days=("date", "count"),
    mean_units=("units", "mean"),
    zero_share=("is_zero", "mean"),
)
profile.describe()

In [ ]:
picks = {
    "fastest": profile["mean_units"].idxmax(),
    "median": profile["mean_units"].sort_values().index[len(profile) // 2],
    "slowest": profile["mean_units"].idxmin(),
}

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, (label, name) in zip(axes, picks.items(), strict=True):
    series = sales[sales["series"] == name].set_index("date")["units"]
    ax.plot(series.index, series.values, linewidth=0.6)
    ax.set_title(f"{label}: {name}")
plt.tight_layout()

In [ ]:
sales["day"] = sales["d"].str[2:].astype(int)

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = sales.groupby(sales["date"].dt.day_name())["units"].mean().reindex(order)
print((by_weekday / sales["units"].mean()).round(2))

In [ ]:
first_day = sales.groupby("series")["day"].min()

for origin in [1829, 1857, 1885, 1913]:
    enough = (first_day <= origin - 56).sum()
    print(f"origin d_{origin}: {enough} of {len(first_day)} series have 8+ weeks of history")

In [ ]:
import numpy as np

wide = sales.pivot(index="series", columns="day", values="units")
wide = wide.reindex(columns=range(1, 1942))
print(wide.shape)
wide.iloc[:3, :5]

In [ ]:
HORIZON = 28
ORIGIN = 1913
MIN_HISTORY = 56

history = wide.loc[:, :ORIGIN]
actual = wide.loc[:, ORIGIN + 1 : ORIGIN + HORIZON]

first_day = history.notna().idxmax(axis=1)
eligible = first_day <= ORIGIN - MIN_HISTORY
history, actual = history[eligible], actual[eligible]

assert not actual.isna().any().any()
print(history.shape, actual.shape)

In [ ]:
def forecast_zero(hist, h):
    return np.zeros((len(hist), h))


def forecast_naive(hist, h):
    last = hist.iloc[:, -1].to_numpy()
    return np.tile(last[:, None], (1, h))


def forecast_seasonal_naive(hist, h):
    last_week = hist.iloc[:, -7:].to_numpy()
    return np.tile(last_week, (1, int(np.ceil(h / 7))))[:, :h]


def forecast_moving_average(hist, h, window=28):
    mean = hist.iloc[:, -window:].mean(axis=1).to_numpy()
    return np.tile(mean[:, None], (1, h))

In [ ]:
def score(forecast, actual):
    error = forecast - actual.to_numpy()
    return {
        "MAE": np.abs(error).mean(),
        "RMSE": np.sqrt((error**2).mean()),
        "bias": error.mean(),
    }


methods = {
    "zero": forecast_zero,
    "naive": forecast_naive,
    "seasonal naive": forecast_seasonal_naive,
    "moving avg 28d": forecast_moving_average,
}

results = {name: score(fn(history, HORIZON), actual) for name, fn in methods.items()}
pd.DataFrame(results).T.round(3)

In [ ]:
FOLDS = [1829, 1857, 1885, 1913]

rows = []
for origin in FOLDS:
    hist = wide.loc[:, :origin]
    act = wide.loc[:, origin + 1 : origin + HORIZON]
    first = hist.notna().idxmax(axis=1)
    keep = first <= origin - MIN_HISTORY
    hist, act = hist[keep], act[keep]
    assert not act.isna().any().any()

    for name, fn in methods.items():
        result = score(fn(hist, HORIZON), act)
        rows.append({"origin": origin, "method": name, "n_series": len(hist), **result})

folds = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "bias"]:
    table = folds.pivot(index="method", columns="origin", values=metric)
    table["mean"] = table.mean(axis=1)
    print(metric)
    print(table.round(3), "\n")

In [ ]:
from ml.metrics import bias, mae, mase, rmse, rmsse

rows = []
for origin in FOLDS:
    hist = wide.loc[:, :origin]
    act = wide.loc[:, origin + 1 : origin + HORIZON]
    keep = hist.notna().idxmax(axis=1) <= origin - MIN_HISTORY
    hist, act = hist[keep], act[keep]
    h, a = hist.to_numpy(dtype=float), act.to_numpy(dtype=float)

    for name, fn in methods.items():
        fc = fn(hist, HORIZON)
        rows.append(
            {
                "origin": origin,
                "method": name,
                "MAE": mae(fc, a),
                "RMSE": rmse(fc, a),
                "MASE": mase(fc, a, h),
                "RMSSE": rmsse(fc, a, h),
            }
        )

scaled = pd.DataFrame(rows)
for metric in ["MASE", "RMSSE"]:
    table = scaled.pivot(index="method", columns="origin", values=metric)
    table["mean"] = table.mean(axis=1)
    print(metric)
    print(table.round(3), "\n")

In [ ]:
from ml.backtest import make_grid, run_backtest, summarize
from ml.baselines import BASELINES

grid = make_grid(sales)
results = run_backtest(grid, BASELINES)
for metric in ["MAE", "RMSE", "MASE", "RMSSE"]:
    print(metric)
    print(summarize(results, metric), "\n")

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, CrostonSBA, TSB, WindowAverage

from ml.backtest import split_fold
from ml.metrics import mae, mase, rmse, rmsse

ORIGIN = 1913
values = grid.to_numpy(dtype=float)
history, actual = split_fold(values, ORIGIN)

observed = ~np.isnan(values[:, :ORIGIN])
first = observed.argmax(axis=1) + 1
names = grid.index[first <= ORIGIN - MIN_HISTORY]
assert len(names) == history.shape[0]

train = sales[(sales["day"] <= ORIGIN) & sales["series"].isin(names)]
train = train[["series", "date", "units"]].rename(
    columns={"series": "unique_id", "date": "ds", "units": "y"}
)
train["y"] = train["y"].astype(float)

sf = StatsForecast(
    models=[
        WindowAverage(window_size=28),
        AutoETS(season_length=7),
        CrostonSBA(),
        TSB(alpha_d=0.2, alpha_p=0.2),
    ],
    freq="D",
)
fc = sf.forecast(df=train, h=HORIZON)
fc.head()

In [ ]:
def to_array(fc, column):
    wide_fc = fc.pivot(index="unique_id", columns="ds", values=column)
    return wide_fc.loc[names].to_numpy().clip(min=0)


rows = []
for model in ["WindowAverage", "AutoETS", "CrostonSBA", "TSB"]:
    forecast = to_array(fc, model)
    rows.append(
        {
            "method": model,
            "MAE": mae(forecast, actual),
            "RMSE": rmse(forecast, actual),
            "MASE": mase(forecast, actual, history),
            "RMSSE": rmsse(forecast, actual, history),
        }
    )

pd.DataFrame(rows).set_index("method").round(3)

In [ ]:
import time

from ml.backtest import make_grid, run_backtest, summarize
from ml.baselines import BASELINES
from ml.stats_baselines import STATS_BASELINES

grid = make_grid(sales)

start = time.time()
stats_results = run_backtest(grid, STATS_BASELINES)
print(f"stats models took {(time.time() - start) / 60:.1f} minutes")

base_results = run_backtest(grid, BASELINES)
all_results = pd.concat([base_results, stats_results], ignore_index=True)

for metric in ["RMSSE", "MASE", "RMSE"]:
    print(metric)
    print(summarize(all_results, metric), "\n")

In [ ]:
cal = pd.read_parquet(DATA / "calendar.parquet").set_index("d")

for origin in (1421, 1785):
    start = cal.loc[f"d_{origin + 1}", "date"].date()
    end = cal.loc[f"d_{origin + 28}", "date"].date()
    print(origin, start, "->", end)

In [ ]:
EXTRA = (1421, 1785)

extra = pd.concat(
    [
        run_backtest(grid, BASELINES, folds=EXTRA),
        run_backtest(grid, STATS_BASELINES, folds=EXTRA),
    ],
    ignore_index=True,
)

print(extra.drop_duplicates("origin")[["origin", "n_series"]])
for metric in ["RMSSE", "RMSE"]:
    print(metric)
    print(summarize(extra, metric), "\n")

In [ ]:
all_results.to_csv(ROOT / "data" / "processed" / "backtest_baselines.csv", index=False)

In [ ]:
pd.concat([all_results, extra]).to_csv(ROOT / "data" / "processed" / "backtest_baselines.csv", index=False)

In [ ]:
combined = all_results.copy()
combined.to_csv(ROOT / "data" / "processed" / "backtest_baselines.csv", index=False)

table = summarize(combined, "RMSSE")
print(table)
print()
print(table.drop(columns="mean").min().round(3))

In [ ]:
from ml.features import origin_features

values = grid.to_numpy(dtype=float)
feats = origin_features(values, origin=1913)
feats.index = grid.index
feats.describe().round(2)

In [ ]:
import time

from ml.backtest import make_grid
from ml.features import FEATURE_COLUMNS, calendar_features, series_states, training_table

prices = make_grid(sales, value="sell_price").to_numpy(dtype=float)
cal = calendar_features(pd.read_parquet(DATA / "calendar.parquet"))
states = series_states(grid.index)

start = time.time()
train = training_table(values, prices, cal, states, cutoff=1913)
print(f"built in {time.time() - start:.0f} seconds")
print(train.shape)
print("last origin:", train["origin"].max(), "| last target day:", train["target_day"].max())
train[FEATURE_COLUMNS].isna().mean().round(3)

In [ ]:
import sys

import lightgbm as lgb
import sklearn

print(sys.executable)
print(sklearn.__version__, lgb.__version__)

In [ ]:
import time

import lightgbm as lgb

from ml.backtest import HORIZON, MIN_HISTORY, split_fold
from ml.features import FEATURE_COLUMNS, build_rows, training_table
from ml.metrics import bias, mae, mase, rmse, rmsse

ORIGIN = 1913
CATEGORICAL = ["dow", "month", "event"]

train_table = training_table(values, prices, cal, states, cutoff=ORIGIN)
assert train_table["target_day"].max() <= ORIGIN

start = time.time()
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(train_table[FEATURE_COLUMNS], train_table["y"], categorical_feature=CATEGORICAL)
print(f"trained in {time.time() - start:.0f} seconds")

In [ ]:
history, actual = split_fold(values, ORIGIN)

observed = ~np.isnan(values[:, :ORIGIN])
first_day = np.where(observed.any(axis=1), observed.argmax(axis=1) + 1, np.inf)
keep = np.flatnonzero(first_day <= ORIGIN - MIN_HISTORY)

rows = build_rows(values, prices, cal, states, origin=ORIGIN, horizon=HORIZON)
rows = rows[rows["series_idx"].isin(keep)].copy()
rows["pred"] = np.clip(model.predict(rows[FEATURE_COLUMNS]), 0, None)

forecast = rows.pivot(index="series_idx", columns="h", values="pred").loc[keep].to_numpy()
assert forecast.shape == actual.shape

print("mean forecast:", forecast.mean().round(3), "| mean actual:", actual.mean().round(3))
pd.Series(
    {
        "MAE": mae(forecast, actual),
        "RMSE": rmse(forecast, actual),
        "bias": bias(forecast, actual),
        "MASE": mase(forecast, actual, history),
        "RMSSE": rmsse(forecast, actual, history),
    }
).round(3)

In [ ]:
from ml.baselines import moving_average

baseline = moving_average(history, HORIZON)
names = grid.index[keep]
category = np.array([n.split("/")[1].split("_")[0] for n in names])

level = pd.Series(np.nanmean(history[:, -56:], axis=1))
speed = pd.qcut(level.rank(method="first"), 3, labels=["slow", "medium", "fast"]).to_numpy()

out = []
for label, groups in {"category": category, "speed": speed}.items():
    for g in np.unique(groups):
        m = groups == g
        out.append(
            {
                "group": f"{label}: {g}",
                "n": int(m.sum()),
                "LightGBM": rmsse(forecast[m], actual[m], history[m]),
                "moving avg": rmsse(baseline[m], actual[m], history[m]),
            }
        )
pd.DataFrame(out).set_index("group").round(3)

In [ ]:
pd.Series(
    model.booster_.feature_importance(importance_type="gain"), index=FEATURE_COLUMNS
).sort_values(ascending=False).round(0)

In [ ]:
import time

from ml.lgbm import backtest

start = time.time()
check = backtest(values, prices, cal, states, origins=(1913,), weighted=False)
print(f"one model took {time.time() - start:.0f} seconds")
check.round(3)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"

from ml.backtest import make_grid, run_backtest, summarize
from ml.baselines import BASELINES
from ml.features import calendar_features, series_states
from ml.lgbm import backtest

sales = pd.read_parquet(DATA / "sales.parquet")
grid = make_grid(sales)
values = grid.to_numpy(dtype=float)
prices = make_grid(sales, value="sell_price").to_numpy(dtype=float)
cal = calendar_features(pd.read_parquet(DATA / "calendar.parquet"))
states = series_states(grid.index)
print(values.shape, prices.shape)

In [ ]:
VALIDATION = (1250, 1700)

start = time.time()
plain = backtest(values, prices, cal, states, VALIDATION, weighted=False, name="LightGBM plain")
weighted = backtest(values, prices, cal, states, VALIDATION, weighted=True, name="LightGBM weighted")
reference = run_backtest(grid, {"moving avg 28d": BASELINES["moving avg 28d"]}, folds=VALIDATION)
print(f"took {(time.time() - start) / 60:.1f} minutes")

validation = pd.concat([plain, weighted, reference], ignore_index=True)
for metric in ["RMSSE", "RMSE", "MASE"]:
    print(metric)
    print(summarize(validation, metric), "\n")

In [ ]:
VALIDATION = (1250, 1700)

start = time.time()
with_wd = backtest(values, prices, cal, states, VALIDATION, weighted=True, name="weighted + weekday")
reference = run_backtest(grid, {"moving avg 28d": BASELINES["moving avg 28d"]}, folds=VALIDATION)
print(f"took {(time.time() - start) / 60:.1f} minutes")

result = pd.concat([with_wd, reference], ignore_index=True)
for metric in ["RMSSE", "RMSE", "MASE"]:
    print(metric)
    print(summarize(result, metric), "\n")

In [ ]:
CANDIDATES = {
    "base": {},
    "small trees": {"num_leaves": 15, "min_child_samples": 300},
    "regularised": {"min_child_samples": 500, "reg_lambda": 10.0},
    "slower + more trees": {"learning_rate": 0.03, "n_estimators": 600},
    "tweedie 1.2": {"objective": "tweedie", "tweedie_variance_power": 1.2},
    "tweedie 1.5": {"objective": "tweedie", "tweedie_variance_power": 1.5},
}

start = time.time()
search = pd.concat(
    [
        backtest(values, prices, cal, states, VALIDATION, weighted=True, params=p, name=name)
        for name, p in CANDIDATES.items()
    ],
    ignore_index=True,
)
print(f"took {(time.time() - start) / 60:.1f} minutes")

for metric in ["RMSSE", "RMSE", "MASE"]:
    print(metric)
    print(summarize(search, metric), "\n")

In [ ]:
from ml.backtest import FOLDS

print("folds:", FOLDS)

start = time.time()
final = backtest(values, prices, cal, states, FOLDS, weighted=True, name="LightGBM")
print(f"took {(time.time() - start) / 60:.1f} minutes")

baselines = pd.read_csv(DATA / "backtest_baselines.csv")
board = pd.concat([baselines, final], ignore_index=True)

for metric in ["RMSSE", "RMSE", "MASE"]:
    print(metric)
    print(summarize(board, metric), "\n")

rmsse = board.pivot(index="method", columns="origin", values="RMSSE")
gap = rmsse.loc["LightGBM"] - rmsse.drop(index="LightGBM").min()
print("LightGBM minus best baseline, per fold (negative = LightGBM wins):")
print(gap.round(3))

board.to_csv(DATA / "backtest_final.csv", index=False)

In [ ]:
from ml.backtest import split_fold
from ml.baselines import moving_average
from ml.features import training_table
from ml.lgbm import fit, forecast
from ml.metrics import rmsse

ORIGIN = 1421
model = fit(training_table(values, prices, cal, states, cutoff=ORIGIN), weighted=True)
history, actual = split_fold(values, ORIGIN)
pred = forecast(model, values, prices, cal, states, origin=ORIGIN)
base = moving_average(history, 28)

dates = pd.read_parquet(DATA / "calendar.parquet").set_index("d")["date"]
per_day = pd.DataFrame(
    {
        "date": [dates[f"d_{ORIGIN + h}"].date() for h in range(1, 29)],
        "actual": actual.mean(axis=0),
        "lgbm": pred.mean(axis=0),
        "moving avg": base.mean(axis=0),
        "lgbm_rmse": np.sqrt(((pred - actual) ** 2).mean(axis=0)),
        "ma_rmse": np.sqrt(((base - actual) ** 2).mean(axis=0)),
    }
).round(2)
print(per_day.to_string())

level = pd.Series(np.nanmean(history[:, -56:], axis=1))
speed = pd.qcut(level.rank(method="first"), 3, labels=["slow", "medium", "fast"]).to_numpy()
for g in ["slow", "medium", "fast"]:
    m = speed == g
    print(g, round(rmsse(pred[m], actual[m], history[m]), 3), round(rmsse(base[m], actual[m], history[m]), 3))

In [ ]:
days = np.arange(28) != 5  # row 5 is 25 December

for g in ["slow", "medium", "fast"]:
    m = speed == g
    print(
        g,
        "| mean actual", round(actual[m].mean(), 3),
        "| lgbm", round(pred[m].mean(), 3),
        "| moving avg", round(base[m].mean(), 3),
        "| RMSSE without Dec 25:",
        round(rmsse(pred[m][:, days], actual[m][:, days], history[m]), 3),
        "vs",
        round(rmsse(base[m][:, days], actual[m][:, days], history[m]), 3),
    )

In [ ]:
from ml.backtest import split_fold
from ml.metrics import coverage, mean_width, pinball
from ml.quantiles import empirical_quantile

VALIDATION = (1250, 1700)
rows = []
for origin in VALIDATION:
    history, actual = split_fold(values, origin)
    for window in (28, 56, 112):
        lo = empirical_quantile(history, 28, 0.1, window)
        hi = empirical_quantile(history, 28, 0.9, window)
        rows.append(
            {
                "origin": origin,
                "window": window,
                "coverage": coverage(lo, hi, actual),
                "width": mean_width(lo, hi),
                "pinball_10": pinball(lo, actual, 0.1),
                "pinball_90": pinball(hi, actual, 0.9),
                "zero_upper": (hi == 0).mean(),
            }
        )

pd.DataFrame(rows).round(3)

In [ ]:
from ml.quantiles import empirical_backtest, quantile_backtest

VALIDATION = (1250, 1700)

start = time.time()
result = pd.concat(
    [
        empirical_backtest(values, VALIDATION, window=112, name="empirical 112d"),
        quantile_backtest(values, prices, cal, states, VALIDATION, name="LightGBM q"),
        quantile_backtest(
            values, prices, cal, states, VALIDATION, weighted=True, name="LightGBM q weighted"
        ),
    ],
    ignore_index=True,
)
print(f"took {(time.time() - start) / 60:.1f} minutes")

print(result.round(3).to_string())

summary = result.groupby("method")[
    ["coverage", "width", "pinball_10", "pinball_90", "zero_upper", "crossed"]
].mean()
summary["pinball_mean"] = (summary["pinball_10"] + summary["pinball_90"]) / 2
summary.round(3)

In [ ]:
from ml.backtest import split_fold
from ml.features import training_table
from ml.metrics import coverage
from ml.quantiles import (
    empirical_quantile,
    fit_quantile,
    interval_breakdown,
    quantile_forecasts,
    speed_groups,
)

VALIDATION = (1250, 1700)
start = time.time()
parts, by_day = [], []

for origin in VALIDATION:
    history, actual = split_fold(values, origin)
    speed = speed_groups(history)
    everyone = np.full(len(speed), "all")

    table = training_table(values, prices, cal, states, cutoff=origin)
    models = [fit_quantile(table, q) for q in (0.1, 0.9)]
    grids, _ = quantile_forecasts(models, values, prices, cal, states, origin)

    intervals = {
        "empirical 112d": (
            empirical_quantile(history, 28, 0.1, 112),
            empirical_quantile(history, 28, 0.9, 112),
        ),
        "LightGBM q": (grids[0], grids[1]),
    }
    for name, (lo, hi) in intervals.items():
        for groups in (everyone, speed):
            part = interval_breakdown(lo, hi, actual, groups)
            part.insert(0, "method", name)
            part.insert(0, "origin", origin)
            parts.append(part)
        by_day.append(
            pd.DataFrame(
                {
                    "method": name,
                    "h": range(1, 29),
                    "coverage": [coverage(lo[:, h], hi[:, h], actual[:, h]) for h in range(28)],
                }
            )
        )

print(f"took {(time.time() - start) / 60:.1f} minutes")

breakdown = pd.concat(parts, ignore_index=True)
columns = ["coverage", "below_lower", "above_upper", "width", "lower_is_zero"]
print(breakdown.groupby(["method", "group"])[columns].mean().round(3))

days = pd.concat(by_day).pivot_table(index="h", columns="method", values="coverage")
print(days.round(3).loc[[1, 7, 14, 21, 28]])

In [ ]:
from ml.coldstart import peer_groups, simulate

groups = peer_groups(grid.index)
VALIDATION = (1250, 1700)

start = time.time()
point_parts, interval_parts = [], []
for origin in VALIDATION:
    for seed in (0, 1, 2):
        point_part, interval_part = simulate(values, prices, cal, states, groups, origin, seed)
        point_parts.append(point_part)
        interval_parts.append(interval_part)
print(f"took {(time.time() - start) / 60:.1f} minutes")

point = pd.concat(point_parts, ignore_index=True)
interval = pd.concat(interval_parts, ignore_index=True)


def show(frame, metric):
    print(metric)
    print(frame.pivot_table(index="method", columns="keep_days", values=metric).round(3), "\n")


for metric in ["RMSSE", "RMSE", "bias"]:
    show(point, metric)
for metric in ["pinball_90", "above_upper", "mean_upper"]:
    show(interval, metric)

In [ ]:
from ml.backtest import FOLDS, split_fold
from ml.features import training_table
from ml.metrics import mean_width, pinball
from ml.quantiles import (
    empirical_quantile,
    fit_quantile,
    interval_breakdown,
    quantile_forecasts,
    speed_groups,
)

start = time.time()
fold_rows, group_parts = [], []

for origin in FOLDS:
    history, actual = split_fold(values, origin)
    speed = speed_groups(history)

    table = training_table(values, prices, cal, states, cutoff=origin)
    models = [fit_quantile(table, q) for q in (0.1, 0.9)]
    grids, crossed = quantile_forecasts(models, values, prices, cal, states, origin)

    intervals = {
        "empirical 112d": (
            empirical_quantile(history, 28, 0.1, 112),
            empirical_quantile(history, 28, 0.9, 112),
        ),
        "LightGBM q": (grids[0], grids[1]),
    }
    for name, (lo, hi) in intervals.items():
        part = interval_breakdown(lo, hi, actual, np.full(len(speed), "all"))
        fold_rows.append(
            {
                "origin": origin,
                "method": name,
                "n_series": history.shape[0],
                "pinball_10": pinball(lo, actual, 0.1),
                "pinball_90": pinball(hi, actual, 0.9),
                "coverage": part.loc[0, "coverage"],
                "below_lower": part.loc[0, "below_lower"],
                "above_upper": part.loc[0, "above_upper"],
                "width": mean_width(lo, hi),
                "crossed": crossed if name == "LightGBM q" else 0.0,
            }
        )
        by_speed = interval_breakdown(lo, hi, actual, speed)
        by_speed.insert(0, "method", name)
        by_speed.insert(0, "origin", origin)
        group_parts.append(by_speed)

print(f"took {(time.time() - start) / 60:.1f} minutes")

final_intervals = pd.DataFrame(fold_rows)
final_intervals["pinball_mean"] = (final_intervals["pinball_10"] + final_intervals["pinball_90"]) / 2
final_intervals.to_csv(DATA / "intervals_final.csv", index=False)

print(final_intervals.round(3).to_string())

mean_table = final_intervals.groupby("method")[
    ["pinball_mean", "pinball_10", "pinball_90", "coverage", "above_upper", "width", "crossed"]
].mean()
print(mean_table.round(3))

pm = final_intervals.pivot(index="origin", columns="method", values="pinball_mean")
print("LightGBM pinball_mean vs empirical, per fold (negative = LightGBM better):")
print(((pm["LightGBM q"] - pm["empirical 112d"]) / pm["empirical 112d"]).round(3))

groups_all = pd.concat(group_parts, ignore_index=True)
print(groups_all.groupby(["method", "group"])[["coverage", "below_lower", "above_upper", "width"]].mean().round(3))

In [ ]:
from ml.coldstart import peer_groups, simulate

peer = peer_groups(grid.index)
start = time.time()
point_parts, interval_parts = [], []
for origin in FOLDS:
    for seed in (0, 1):
        point_part, interval_part = simulate(
            values, prices, cal, states, peer, origin, seed, k0s=(7,)
        )
        point_parts.append(point_part)
        interval_parts.append(interval_part)
print(f"took {(time.time() - start) / 60:.1f} minutes")

cold_point = pd.concat(point_parts, ignore_index=True)
cold_interval = pd.concat(interval_parts, ignore_index=True)
cold_point.to_csv(DATA / "coldstart_point_final.csv", index=False)
cold_interval.to_csv(DATA / "coldstart_interval_final.csv", index=False)

for frame, metric in [
    (cold_point, "RMSSE"),
    (cold_interval, "above_upper"),
    (cold_interval, "pinball_90"),
]:
    print(metric)
    print(frame.pivot_table(index="method", columns="keep_days", values=metric).round(3), "\n")